In [1]:
!pip install pyspark


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, desc


In [3]:
spark = SparkSession.builder \
    .appName("Lab2_ArquitecturaDatos") \
    .master("local[*]") \
    .getOrCreate()

print("Motor Spark Iniciado. Versión:", spark.version)


Motor Spark Iniciado. Versión: 4.0.2


In [6]:
df_raw = spark.read.json("streaming_logs.json")

In [7]:
df_raw.printSchema()


root
 |-- device: string (nullable = true)
 |-- seconds_played: long (nullable = true)
 |-- song_info: struct (nullable = true)
 |    |-- artist: string (nullable = true)
 |    |-- track_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- user_id: string (nullable = true)



In [8]:
df_flatten = df_raw.select(
    col("user_id"),
    col("timestamp"),
    col("song_info.artist").alias("artista"),
    col("device"),
    col("seconds_played").alias("duracion")
)

df_flatten.show(3)


+-------+--------------------+---------+-------+--------+
|user_id|           timestamp|  artista| device|duracion|
+-------+--------------------+---------+-------+--------+
|user_31|2026-01-18T16:35:...|Bad Bunny| iPhone|     108|
|user_31|2026-01-25T16:35:...|      BTS| iPhone|     131|
|user_31|2026-01-22T16:35:...|  Shakira|Android|     137|
+-------+--------------------+---------+-------+--------+
only showing top 3 rows


In [9]:
total_raw = df_flatten.count()


In [10]:
df_clean = df_flatten.filter(
    (col("duracion").isNotNull()) &
    (col("duracion") > 0)
)


In [11]:
total_clean = df_clean.count()

print("Registros eliminados:", total_raw - total_clean)


Registros eliminados: 3393


In [12]:
df_clean.createOrReplaceTempView("tabla_reproducciones")


In [13]:
query_top_artistas = """
    SELECT 
        artista,
        COUNT(*) as total_reproducciones,
        ROUND(SUM(duracion) / 3600, 2) as horas_totales
    FROM tabla_reproducciones
    GROUP BY artista
    ORDER BY total_reproducciones DESC
    LIMIT 3
"""


In [14]:
spark.sql(query_top_artistas).show()


+------------+--------------------+-------------+
|     artista|total_reproducciones|horas_totales|
+------------+--------------------+-------------+
|Taylor Swift|                7835|       357.18|
|       Drake|                7812|       358.21|
|         BTS|                7780|       357.61|
+------------+--------------------+-------------+



In [15]:
spark.stop()
